# QSVT toolkit — interactive demo

A from-scratch **C++** implementation of Quantum Singular Value Transformation, driven from Python via the `qsvt_native` bindings (NumPy in/out).

It compiles a matrix function f(A) into a quantum circuit and runs the three canonical QSVT applications — matrix inversion, Hamiltonian simulation, and eigenvalue thresholding — each validated against exact linear algebra.

## Setup

Build the bindings first:

```bash
cmake -S . -B build -DBUILD_PYTHON=ON && cmake --build build --target qsvt_native
```

then make `qsvt_native` importable, e.g. `PYTHONPATH=build/bindings`.

In [ ]:
import numpy as np
import qsvt_native as q

def random_hermitian(eigs, seed):
    rng = np.random.default_rng(seed); n = len(eigs)
    a = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))
    U, _ = np.linalg.qr(a)
    return U @ np.diag(eigs) @ U.conj().T

## 1. Compile a unitary to native gates + OpenQASM

`shannon_decompose` runs the recursive cosine-sine / quantum-Shannon decomposition and returns CNOT/depth counts, the reconstruction error, and an OpenQASM 2.0 program.

In [ ]:
rng = np.random.default_rng(2)
U, _ = np.linalg.qr(rng.standard_normal((8, 8)) + 1j * rng.standard_normal((8, 8)))
d = q.shannon_decompose(U)
print(d['cnot'], 'CNOTs, depth', d['depth'], ', recon error', d['reconstruction_error'])
print('\n'.join(d['qasm'].splitlines()[:6]))

## 2. Matrix inversion

Apply ~ c/A to a well-conditioned Hermitian operator with a degree-25 QSVT circuit (regularised inverse c*x/(x^2+eps)). The Hermitian part of the QSVT block is the realised real transform.

In [ ]:
A = random_hermitian([0.4, 0.6, 0.8, 1.0], 3)
delta = 0.4; eps = (delta / 4) ** 2; c = 0.9 * 2 * np.sqrt(eps)
prog = q.compile_matrix_function(A, lambda x: c * x / (x * x + eps), 25)
block = q.qsvt_block(A, prog['phases']); realized = 0.5 * (block + block.conj().T)
lam, V = np.linalg.eigh(A); target = V @ np.diag(c / lam) @ V.conj().T
print(prog['num_qubits'], 'qubits,', prog['cnot'], 'CNOTs')
print('|| realized - c A^-1 || =', np.linalg.norm(realized - target))

## 3. Hamiltonian simulation e^{-iHt}

e^{-iHt} = cos(tH) - i sin(tH); the angle solver fits cos/sin (Jacobi-Anger) and QSVT applies them.

In [ ]:
H = random_hermitian([0.6, -0.4, 0.3, -0.7], 11); t = 2.5
approx = q.hamiltonian_evolution(H, t, 21)
lamH, VH = np.linalg.eigh(H); exact = VH @ np.diag(np.exp(-1j * t * lamH)) @ VH.conj().T
print('|| e^-iHt_QSVT - exact || =', np.linalg.norm(approx - exact))

## 4. Eigenvalue thresholding (spectral projector)

Project onto eigenvalues above a threshold via sign(H - mu): Pi = (I + sign(H - mu)) / 2.

In [ ]:
Hp = random_hermitian([-0.7, -0.4, 0.4, 0.8], 5)
P = q.spectral_projector(Hp, mu=0.0, w=0.1, degree=25)
lamP, VP = np.linalg.eigh(Hp)
exactP = VP @ np.diag([1.0 if l > 0 else 0.0 for l in lamP]) @ VP.conj().T
print('|| P_QSVT - exact projector || =', np.linalg.norm(P - exactP))

---
Every result above is checked against exact linear algebra. The same circuits are also runnable on the Qrack GPU statevector simulator from the C++ side.